# 0s · **팀 공유용 결과 묶음** — `transfer`

eval 이 끝난 transfer 결과를 **표(CSV) + 그림(PNG) + 요약(MD)** 으로 뽑고 **zip** 으로 묶는다.
그 zip 하나만 팀원에게 보내면 된다.

| 산출물 | 내용 |
|---|---|
| `sr.csv` | SR — mean ± std(20 run), Wilson 95% CI, seed별 |
| `smoothness.csv` | 떨림 — 경계/내부 jerk, **contrast**, ratio, SPARC |
| `figs/sr.png` | SR 막대 + 개별 run 점 |
| `figs/boundary_jerk_profile.png` | **경계정렬 jerk 프로파일** (핵심 그림) |
| `figs/jerk_bars.png` | 경계 vs 내부 떨림 |
| `summary.md` | 카톡/메일에 그대로 붙여넣을 요약 |

결과가 있는 모델만 자동으로 포함된다(아직 안 돌린 건 조용히 빠짐).


In [ ]:
import sys, csv, json, shutil, statistics as st
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)
import numpy as np
import smooth_metrics as sm

TASK  = cf.SHORT_SIM                    # 'transfer'
SEEDS = cf.MAIN_SEEDS                   # [0,1,2,3]
REPS  = list(range(cf.EVAL_REPEATS))    # 5
N_EP  = cf.EVAL_N_EP                    # 500
K     = 100                             # chunk size (경계 = 100, 200, ...)
FPS   = cf.fps_of(TASK)                 # 50

OUT = cf.OUTPUT_BASE / 'share' / TASK
(OUT / 'figs').mkdir(parents=True, exist_ok=True)

# 결과가 실제로 있는 모델만 (없는 건 조용히 빠짐)
ALL = cf.FINAL_TAGS + [t for t in cf.ABLATION if t not in cf.FINAL_TAGS]
TAGS = [t for t in ALL
        if cf.sr_over_reps(t, task=TASK, seeds=SEEDS, reps=REPS)['mean'] is not None]

print('task :', TASK, '| ckpt', f'{cf.CKPT_STEP:,}', '| seeds', SEEDS, '| reps', REPS, '| ep', N_EP)
print('결과 있는 모델:', TAGS)
missing = [t for t in ALL if t not in TAGS]
if missing:
    print('결과 없음(제외):', missing)
print('\n저장 위치:', OUT)

## 1) SR 표


In [ ]:
# ── 1) SR 표 ────────────────────────────────────────────────────────────────
sr_rows = []
for t in TAGS:
    a = cf.sr_over_reps(t, task=TASK, seeds=SEEDS, reps=REPS)
    n_ep = a['n_runs'] * N_EP
    lo, hi = cf.wilson_ci(int(round(a['mean'] / 100 * n_ep)), n_ep)
    row = {'tag': t, 'model': cf.v23.MODEL_LABELS.get(t, t),
           'SR_mean': round(a['mean'], 2), 'SR_std': round(a['std'], 2),
           'CI_lo': round(lo * 100, 1), 'CI_hi': round(hi * 100, 1),
           'n_run': a['n_runs'], 'n_episodes': n_ep}
    for s, v in a['per_seed'].items():
        row[f'seed{s}'] = round(v, 1)
    sr_rows.append(row)

with open(OUT / 'sr.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(sr_rows[0]))
    w.writeheader()
    w.writerows(sr_rows)

print(f"{'MODEL':<32}{'SR (mean±std)':>17}{'95% CI':>17}{'runs':>6}")
print('-' * 72)
for r in sr_rows:
    srtxt = '%.1f ± %.1f' % (r['SR_mean'], r['SR_std'])
    citxt = '[%.1f, %.1f]' % (r['CI_lo'], r['CI_hi'])
    print(f"{r['model']:<32}{srtxt:>17}{citxt:>17}{r['n_run']:>6}")
print('\n저장:', OUT / 'sr.csv')

## 1b) ★ **표 이미지** — 모델 × step, eval 1~5 + avg

카톡에 그대로 올릴 수 있는 **표 그림**(PNG). 각 칸 = 그 eval 의 SR(%),
seed 가 여러 개면 그 rep 의 **seed 평균**. 마지막 열 = 5회 평균.

- `STEPS_SHOWN` 에 여러 step 을 주면 행이 `100k` / `150k` 로 나뉜다.
  단 **그 step 을 실제로 eval 해둔 경우**만 (안 돌린 step 은 자동으로 빠짐).
- 100k 도 보고 싶으면: `cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, gpus=GPUS,
  n_episodes=N_EP, step=100_000)` 로 100k 체크포인트를 먼저 평가할 것.
  (결과는 `100k_rep*` 로 따로 저장되어 150k 결과를 덮어쓰지 않는다)


In [ ]:
# ── 표 이미지 (스크린샷 형태) ────────────────────────────────────────────────
STEPS_SHOWN = [cf.CKPT_STEP]              # 100k 도 평가했다면 [100_000, cf.CKPT_STEP]
TABLE_TAGS  = [t for t in ('act', 'act_te', 'diffusion', 'smolvla', 'acm2', 'acm', 'ours')
               if t in TAGS]              # 결과 있는 것만, 논문 순서대로

rows = cf.sr_grid_table(TABLE_TAGS, task=TASK, seeds=SEEDS, reps=REPS, steps=STEPS_SHOWN,
                        png_path=OUT / 'figs' / 'table_sr.png',
                        csv_path=OUT / 'table_sr.csv')


## 2) 떨림(매끄러움) 표
`aggregate_smoothness` 는 rep 5개의 action(.pt) 을 전부 pool 해서 계산한다.


In [ ]:
# ── 2) 떨림(매끄러움) 표 ─────────────────────────────────────────────────────
# aggregate_smoothness 는 키에 _mean / _std 접미사가 붙는다.
trajs = {t: [tr for s in SEEDS for tr in cf.action_trajs(t, s, TASK, reps=REPS)] for t in TAGS}

jk_rows = []
for t in TAGS:
    tr = trajs[t]
    if not tr:
        print(f'  {t}: action(.pt) 없음 — 건너뜀')
        continue
    m = sm.aggregate_smoothness(tr, chunk_size=K, fs=FPS)
    jk_rows.append({
        'tag': t, 'model': cf.v23.MODEL_LABELS.get(t, t), 'n_episodes': m['n_episodes'],
        'boundary_jerk': round(m['boundary_jerk_mean'], 5),
        'interior_jerk': round(m['interior_jerk_mean'], 5),
        'contrast': round(m['boundary_jerk_contrast_mean'], 5),   # ★ 경계 튐 (0 에 가까울수록 좋음)
        'ratio': round(m['boundary_jerk_ratio_mean'], 3),         # 1 에 가까울수록 좋음
        'jerk_rms': round(m['jerk_rms_mean'], 5),
        'SPARC': round(m['sparc_mean'], 3),                       # ★ 0 에 가까울수록 매끄러움
    })

if jk_rows:
    with open(OUT / 'smoothness.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(jk_rows[0]))
        w.writeheader()
        w.writerows(jk_rows)
    print(f"{'MODEL':<32}{'b-jerk':>10}{'i-jerk':>10}{'contrast':>11}{'ratio':>8}{'SPARC':>9}")
    print('-' * 80)
    for r in jk_rows:
        print(f"{r['model']:<32}{r['boundary_jerk']:>10.4f}{r['interior_jerk']:>10.4f}"
              f"{r['contrast']:>11.4f}{r['ratio']:>8.2f}{r['SPARC']:>9.2f}")
    print('\ncontrast = 경계 jerk − 내부 jerk (0 에 가까울수록 경계에서 안 튐)')
    print('저장:', OUT / 'smoothness.csv')
else:
    print('action(.pt) 이 하나도 없다 — eval 이 RECORD_DIR 로 기록했는지 확인')

## 3) 그림


In [ ]:
# ── 3) 그림 ─────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'font.size': 13, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
# 그림 안 글자는 전부 영문 — 클러스터엔 한글 폰트가 없어서(findfont: Malgun Gothic not found)
# 한글을 쓰면 □□ 로 깨진다. 논문 그림도 어차피 영문.
plt.rcParams['axes.unicode_minus'] = False

F = OUT / 'figs'

# (a) SR 막대 + 개별 run 점
fig, ax = plt.subplots(figsize=(11, 5))
for i, r in enumerate(sr_rows):
    a = cf.sr_over_reps(r['tag'], task=TASK, seeds=SEEDS, reps=REPS)
    ax.bar(i, a['mean'], yerr=a['std'], color=cf.COLOR.get(r['tag'], '#333'),
           alpha=0.85, capsize=5, width=0.62)
    ax.scatter([i] * len(a['all']), a['all'], s=12, color='k', alpha=0.35, zorder=3)
ax.set_xticks(range(len(sr_rows)))
ax.set_xticklabels([r['model'] for r in sr_rows], rotation=20, ha='right')
ax.set_ylabel('Success rate (%)')
ax.set_title(f'{TASK} @150k — {len(REPS)} reps x {len(SEEDS)} seeds x {N_EP} ep '
             f'(dots = individual runs)', fontweight='bold')
fig.savefig(F / 'sr.png'); plt.show()

# (b) 경계정렬 jerk 프로파일 ★헤드라인
if jk_rows:
    fig, ax = plt.subplots(figsize=(9, 5))
    W = 12
    for r in jk_rows:
        t = r['tag']
        prof = []
        for traj in trajs[t]:
            jm = sm.jerk_magnitude(traj)                 # (T-?,)
            for b in range(K, len(jm) - W, K):           # 청크 경계마다
                prof.append(jm[b - W:b + W])
        if not prof:
            continue
        p = np.mean(np.stack(prof), axis=0)
        ax.plot(np.arange(-W, W), p, label=cf.v23.MODEL_LABELS.get(t, t),
                color=cf.COLOR.get(t, '#333'), lw=2)
    ax.axvline(0, ls='--', c='r', alpha=0.6)
    ax.set(xlabel='steps relative to chunk boundary (0 = boundary)', ylabel='|jerk|',
           title='Boundary-aligned jerk profile')
    ax.legend(fontsize=10)
    fig.savefig(F / 'boundary_jerk_profile.png'); plt.show()

    # (c) 경계 vs 내부 jerk 막대
    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(jk_rows))
    ax.bar(x - 0.2, [r['boundary_jerk'] for r in jk_rows], 0.4, label='boundary', color='#d62728', alpha=0.85)
    ax.bar(x + 0.2, [r['interior_jerk'] for r in jk_rows], 0.4, label='interior', color='#888', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([r['model'] for r in jk_rows], rotation=20, ha='right')
    ax.set_ylabel('|jerk|')
    ax.set_title('Jerk at chunk boundaries vs interior', fontweight='bold')
    ax.legend()
    fig.savefig(F / 'jerk_bars.png'); plt.show()

print('그림 저장:', F)

## 3b) ★ **Δaction vs timestep** — 청크 경계에서 튀는가

실행된 액션의 **1차 차분** `Δa_t = a_t − a_{t-1}`. 경계 불연속이 **스파이크**로 드러난다.

- 빨간 점선 = 청크 경계 (t = 100, 200, 300 …)
- **에피소드 전 구간** 표시, **모델 간 y축·action 차원을 동일하게 고정** → 눈으로 바로 비교 가능
  (모델마다 축이 다르면 작은 떨림이 커 보이는 착시가 생긴다)
- 위: 모델별 subplot / 아래: `ours` vs `acm` 한 축에 겹쳐 그린 것


In [ ]:
# ── Δaction vs timestep (rep/seed pool 중 대표 에피소드) ─────────────────────
N_DIMS = 3          # 변동 큰 상위 N개 action 차원만 (None 이면 전부)

# 대표 에피소드 = 각 모델에서 가장 긴 궤적 (전 구간을 보기 위해)
rep_traj = {t: max(trajs[t], key=len) for t in TAGS if trajs.get(t)}
if not rep_traj:
    print('action(.pt) 없음')
else:
    # 비교 차원은 **모델 공통**으로 고정 (모델마다 다른 dim 을 그리면 비교가 안 됨)
    var = np.mean([np.asarray(tr).var(axis=0) for tr in rep_traj.values()], axis=0)
    dims = np.argsort(var)[::-1][:N_DIMS] if N_DIMS else np.arange(len(var))

    # Δ[i] = a[i+1] − a[i]  → 청크 경계 b(새 청크의 첫 액션)를 만드는 전이는 Δ[b-1] 이다.
    D = {t: np.diff(np.asarray(tr)[:, dims], axis=0) for t, tr in rep_traj.items()}
    BND = lambda n: [b - 1 for b in range(K, n + 1, K)]      # Δ 좌표계의 경계 위치
    ymax = max(np.abs(d).max() for d in D.values()) * 1.05
    Tmax = max(len(d) for d in D.values())

    n = len(D)
    fig, axes = plt.subplots(n, 1, figsize=(12, 2.6 * n), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (t, d) in zip(axes, D.items()):
        for j, dim in enumerate(dims):
            ax.plot(d[:, j], lw=0.8, alpha=0.85, label=f'dim {dim}')
        for b in BND(len(d)):
            ax.axvline(b, color='r', ls='--', lw=0.8, alpha=0.5)
        ax.set_ylim(-ymax, ymax)
        ax.set_xlim(0, Tmax)
        ax.set_ylabel('delta action')
        ax.set_title(cf.v23.MODEL_LABELS.get(t, t), loc='left', fontsize=12, fontweight='bold')
        ax.legend(fontsize=8, ncol=len(dims), loc='upper right')
    axes[-1].set_xlabel('timestep   (red dashed = chunk boundary, K=%d)' % K)
    fig.suptitle('Delta-action vs timestep (shared y-limits and action dims)', fontweight='bold')
    fig.tight_layout()
    fig.savefig(F / 'delta_action.png')
    plt.show()

    # ours vs acm 겹쳐 그리기 (있을 때만)
    pair = [t for t in ('acm', 'ours') if t in D]
    if len(pair) == 2:
        dim0 = 0
        fig, ax = plt.subplots(figsize=(12, 4))
        for t in pair:
            ax.plot(D[t][:, dim0], lw=1.0, alpha=0.9,
                    color=cf.COLOR.get(t, '#333'), label=cf.v23.MODEL_LABELS.get(t, t))
        for b in BND(Tmax):
            ax.axvline(b, color='r', ls='--', lw=0.8, alpha=0.5)
        ax.set(xlabel='timestep', ylabel=f'delta action (dim {dims[dim0]})',
               title='acm (carry off) vs ours — delta action at chunk boundaries')
        ax.set_xlim(0, Tmax)
        ax.legend()
        fig.savefig(F / 'delta_action_ours_vs_acm.png')
        plt.show()

    print('저장:', F / 'delta_action.png')


## 3c) ★ **20칸 그리드** — seed × rep (eval 1~5) 별로 어땠나

**행 = 학습 seed(4개), 열 = eval 반복 rep 1~5** → 한 모델당 **20칸**.
각 칸이 그 run 의 **Δaction vs timestep** (빨간 점선 = 청크 경계).

- 20 run 이 **전부 같은 y축·같은 action 차원**으로 그려진다 → 칸끼리 바로 비교 가능
- 각 칸 제목에 그 run 의 **SR** 표시 → 떨림과 성공률을 같이 볼 수 있음
- 같이 나오는 `sr_grid_*.png` 는 같은 20칸을 **SR 숫자 표**로 (색 = 높을수록 진함)


In [ ]:
# ── 20칸 그리드: seed(행) x rep(열) ──────────────────────────────────────────
GRID_TAGS = [t for t in ('ours', 'acm') if t in TAGS] or TAGS[:1]   # 필요하면 TAGS 로 바꿀 것

for t in GRID_TAGS:
    # (seed, rep) 별 대표 궤적 + SR
    cell_traj, cell_sr = {}, {}
    for s in SEEDS:
        for r in REPS:
            tr = cf.action_trajs(t, s, TASK, reps=[r])
            if tr:
                cell_traj[(s, r)] = max(tr, key=len)
            cell_sr[(s, r)] = cf.rep_sr(t, s, TASK, r)
    if not cell_traj:
        print(f'{t}: action(.pt) 없음 — 건너뜀')
        continue

    # 축·차원은 20칸 전체 공통 (칸마다 다르면 비교가 안 된다)
    var = np.mean([np.asarray(v).var(axis=0) for v in cell_traj.values()], axis=0)
    dims = np.argsort(var)[::-1][:N_DIMS] if N_DIMS else np.arange(len(var))
    Dg = {k: np.diff(np.asarray(v)[:, dims], axis=0) for k, v in cell_traj.items()}
    ymax = max(np.abs(d).max() for d in Dg.values()) * 1.05
    Tmax = max(len(d) for d in Dg.values())

    fig, axes = plt.subplots(len(SEEDS), len(REPS), figsize=(4.0 * len(REPS), 2.4 * len(SEEDS)),
                             sharex=True, sharey=True)
    axes = np.atleast_2d(axes)
    for i, s in enumerate(SEEDS):
        for j, r in enumerate(REPS):
            ax = axes[i, j]
            d = Dg.get((s, r))
            if d is None:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
            else:
                for k in range(d.shape[1]):
                    ax.plot(d[:, k], lw=0.7, alpha=0.85)
                for b in range(K - 1, len(d), K):        # Δ 좌표계 경계
                    ax.axvline(b, color='r', ls='--', lw=0.7, alpha=0.5)
            sr = cell_sr.get((s, r))
            ax.set_title(f'seed{s} · eval{r + 1}' + (f'   SR {sr:.1f}%' if sr is not None else ''),
                         fontsize=10, loc='left')
            ax.set_ylim(-ymax, ymax)
            ax.set_xlim(0, Tmax)
    for i, s in enumerate(SEEDS):
        axes[i, 0].set_ylabel('delta action')
    for j in range(len(REPS)):
        axes[-1, j].set_xlabel('timestep')
    fig.suptitle(f'{cf.v23.MODEL_LABELS.get(t, t)} — {TASK} @150k : '
                 f'{len(SEEDS)} seeds x {len(REPS)} evals = {len(SEEDS) * len(REPS)} runs '
                 f'(red dashed = chunk boundary, shared y-limits)', fontweight='bold')
    fig.tight_layout()
    fig.savefig(F / f'delta_grid_{t}.png')
    plt.show()

    # 같은 20칸을 SR 숫자 표로
    M = np.full((len(SEEDS), len(REPS)), np.nan)
    for i, s in enumerate(SEEDS):
        for j, r in enumerate(REPS):
            v = cell_sr.get((s, r))
            if v is not None:
                M[i, j] = v
    fig, ax = plt.subplots(figsize=(1.5 * len(REPS) + 2, 1.0 * len(SEEDS) + 1.6))
    im = ax.imshow(M, cmap='YlGn', vmin=np.nanmin(M), vmax=np.nanmax(M), aspect='auto')
    for i in range(len(SEEDS)):
        for j in range(len(REPS)):
            v = M[i, j]
            ax.text(j, i, '-' if np.isnan(v) else f'{v:.1f}', ha='center', va='center',
                    fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(REPS)))
    ax.set_xticklabels([f'eval {r + 1}' for r in REPS])
    ax.set_yticks(range(len(SEEDS)))
    ax.set_yticklabels([f'seed {s}' for s in SEEDS])
    row_mean = np.nanmean(M, axis=1)
    for i, m in enumerate(row_mean):
        ax.text(len(REPS) - 0.35, i, f'  (avg {m:.1f})', va='center', fontsize=9, color='#444')
    ax.set_title(f'{cf.v23.MODEL_LABELS.get(t, t)} — SR per run (%)   '
                 f'mean {np.nanmean(M):.1f} ± {np.nanstd(M):.1f}', fontweight='bold')
    ax.grid(False)
    fig.colorbar(im, ax=ax, label='SR (%)')
    fig.tight_layout()
    fig.savefig(F / f'sr_grid_{t}.png')
    plt.show()

    print(f'{t}: delta_grid_{t}.png / sr_grid_{t}.png 저장')


## 4) 요약 — 그대로 복붙


In [ ]:
# ── 4) 요약(카톡/메일 붙여넣기용) → summary.md ───────────────────────────────
def get(rows, tag, key):
    for r in rows:
        if r['tag'] == tag:
            return r[key]
    return None

lines = []
lines.append(f'# transfer 결과 ({TASK}) — 150k 체크포인트')
lines.append('')
lines.append(f'- 프로토콜: 150k step 학습(lr 1e-5 고정) → **150k 체크포인트 1개**를 '
             f'**{len(REPS)}회 반복 × {N_EP} 에피소드** 평가')
lines.append(f'- rep 마다 env seed 를 바꿔 평가(1000, 1100, …) → 학습 seed({len(SEEDS)}개) 분산과 분리')
lines.append(f'- 모델당 **{len(SEEDS)} seed × {len(REPS)} rep = {len(SEEDS)*len(REPS)} run '
             f'= {len(SEEDS)*len(REPS)*N_EP:,} 에피소드**')
lines.append('')
lines.append('## 성공률 (SR)')
lines.append('')
lines.append('| 모델 | SR (mean ± std) | 95% CI |')
lines.append('|---|---|---|')
for r in sr_rows:
    lines.append(f"| {r['model']} | {r['SR_mean']:.1f} ± {r['SR_std']:.1f} | "
                 f"[{r['CI_lo']:.1f}, {r['CI_hi']:.1f}] |")
lines.append('')

# 핵심 비교
o, a, act = get(sr_rows, 'ours', 'SR_mean'), get(sr_rows, 'acm', 'SR_mean'), get(sr_rows, 'act', 'SR_mean')
if o is not None and a is not None:
    lines.append(f'- **ours vs acm(carry off)**: {o:.1f} vs {a:.1f} → **{o - a:+.1f}p**')
if o is not None and act is not None:
    lines.append(f'- **ours vs ACT**: {o:.1f} vs {act:.1f} → **{o - act:+.1f}p**')
lines.append('')

if jk_rows:
    lines.append('## 매끄러움 (떨림)')
    lines.append('')
    lines.append('| 모델 | 경계 jerk | 내부 jerk | **contrast** | ratio | SPARC |')
    lines.append('|---|---|---|---|---|---|')
    for r in jk_rows:
        lines.append(f"| {r['model']} | {r['boundary_jerk']:.4f} | {r['interior_jerk']:.4f} | "
                     f"**{r['contrast']:.4f}** | {r['ratio']:.2f} | {r['SPARC']:.2f} |")
    lines.append('')
    lines.append('- **contrast = 경계 jerk − 내부 jerk.** 0 에 가까울수록 청크 경계에서 안 튄다(= MOSAIC 효과).')
    lines.append('- ratio 는 1 에 가까울수록, SPARC 는 0 에 가까울수록 매끄럽다.')
    lines.append('')

lines.append('## 파일')
lines.append('- `sr.csv` — SR (seed별 포함)')
lines.append('- `figs/table_sr.png` — **표 이미지** (모델 × step, eval 1~5 + avg)')
lines.append('- `table_sr.csv` — 그 표의 원본 수치')
if jk_rows:
    lines.append('- `smoothness.csv` — 떨림 지표')
lines.append('- `figs/sr.png` — SR 막대(점 = 개별 run)')
if jk_rows:
    lines.append('- `figs/boundary_jerk_profile.png` — **경계정렬 jerk 프로파일**(핵심 그림)')
    lines.append('- `figs/jerk_bars.png` — 경계 vs 내부 떨림')
    lines.append('- `figs/delta_action.png` — **Δaction vs timestep** (경계 스파이크)')
    lines.append('- `figs/delta_action_ours_vs_acm.png` — 같은 축에 겹쳐 비교')
    lines.append('- `figs/delta_grid_*.png` — **20칸(seed × eval 1~5) Δaction 그리드**')
    lines.append('- `figs/sr_grid_*.png` — 같은 20칸의 SR 숫자 표')

txt = '\n'.join(lines) + '\n'
(OUT / 'summary.md').write_text(txt, encoding='utf-8')
print(txt)

## 5) zip


In [ ]:
# ── 5) zip 으로 묶기 → 이 파일을 팀원에게 보내면 됨 ──────────────────────────
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'{TASK}_150k'), 'zip', root_dir=OUT)
print('보낼 파일:', zip_path)
print()
for p in sorted(Path(OUT).rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')